# Tweety C# / IKVM - Logiques de Base (Port .NET du notebook Python)

**Navigation** : [<- Tweety-1-Setup](Tweety-1-Setup.ipynb) | [Index serie](Tweety-1-Setup.ipynb) | [Tweety-3-Advanced-Logics ->](Tweety-3-Advanced-Logics.ipynb)

> Ce notebook est le **port C# / .NET** de [Tweety-2-Basic-Logics.ipynb](Tweety-2-Basic-Logics.ipynb).
> Au lieu d'appeler Tweety depuis Python via JPype + JVM, on execute directement les
> classes Tweety **recompilees en bytecode .NET via IKVM** - aucune JVM, aucun JDK requis
> au runtime. Voir le notebook Python pour le contexte theorique complet.

## Objectifs pedagogiques

1. Charger la DLL Tweety (cluster `pl`) recompilee via IKVM 8.15
2. Construire des formules propositionnelles (`Proposition`, `Negation`, `Conjunction`, `Disjunction`, `Implication`)
3. Parser des formules depuis une chaine avec `PlParser`
4. Manipuler une base de croyances (`PlBeliefSet`) et explorer les mondes possibles (`PossibleWorld`)

## Prerequis

- Le notebook Python `Tweety-2-Basic-Logics.ipynb` pour les notions (logique propositionnelle, mondes possibles)
- Le runtime est **tout .NET** : aucune installation Java/JDK n'est necessaire


## Comment Tweety tourne en .NET (sans JVM)

Tweety est une bibliotheque **Java**. Pour l'utiliser depuis C#, on recompile son bytecode
Java en **bytecode .NET** grace a [IKVM](https://github.com/ikvm-revived/ikvm) :

1. **Recompilation** : le bytecode Tweety 1.30 (Java 15) est recompile vers Java 8 (compatible IKVM 8.15, qui cible Java 8 SE), puis converti en une DLL .NET via `<IkvmReference>`.
2. **Fat-jar Maven shade** : on assemble `pl` + ses dependances transitives (`fol`, `logics-commons`, `math`, `commons`, `commons-math3`, `ojalgo`, `sat4j`) en un unique fat-jar Maven coheren, qui preserve les metadonnees cross-module (un zip-merge artisanal les casserait).
3. **Runtime** : la DLL `org.tweetyproject.tweety-pl.dll` (placee a cote de ce notebook) est referencee via `#r` ; IKVM fournit l'execution de la JVM en .NET. La recette de build complete (POM shade + csproj) est dans [`dotnet-build/`](dotnet-build/).

La cellule suivante configure ce runtime - **a executer avant toute utilisation de Tweety**.


### 1. Configuration du runtime IKVM

> Cette cellule restaure les paquets NuGet IKVM et assemble le home IKVM. ~30-60 s la premiere fois.


In [1]:
#r "nuget: IKVM, 8.15.0"
#r "nuget: IKVM.Image, 8.15.0"
#r "nuget: IKVM.Image.runtime.win-x64, 8.15.0"


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages IKVM, 8.15.0 IKVM.Image, 8.15.0

In [2]:
using System.IO;

string ikvmVer = "8.15.0", ikvmRid = "win-x64";
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome    = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);

void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t));
        File.Copy(f, t, overwrite: true);
    }
}

if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);
    IkvmCopyMerge(ikvmArchDir, ikvmHome);
}
AppContext.SetData("IKVM.Home", ikvmHome);

bool tzdbOk = File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat"));
Console.WriteLine($"IKVM 8.15.0 pret (home=ikvm-home-{ikvmVer}-{ikvmRid}, tzdb={tzdbOk})");


IKVM 8.15.0 pret (home=ikvm-home-8.15.0-win-x64, tzdb=True)


### 2. Chargement de la DLL Tweety

On reference la DLL recompilee. Le cluster `pl` expose les espaces de noms
`org.tweetyproject.logics.pl.syntax`, `org.tweetyproject.logics.pl.parser`, etc.


In [3]:
#r "org.tweetyproject.tweety-pl.dll"


In [4]:
// Constat 3 (#5039) : la directive #r est silencieuse par defaut en .NET Interactive.
// On verifie que la DLL IKVM/Tweety referencee est bien presente et lisible (metadata).
using System.Reflection;
var tweetyDll = "org.tweetyproject.tweety-pl.dll";
var an = AssemblyName.GetAssemblyName(tweetyDll);
Console.WriteLine($"Tweety (IKVM) reference chargee : {an.Name} v{an.Version} ({new FileInfo(tweetyDll).Length / 1024 / 1024:F1} Mo).");

Tweety (IKVM) reference chargee : org.tweetyproject.tweety-pl v1.30.0.0 (7,0 Mo).


### 3. Logique propositionnelle - construction manuelle de formules

On construit des formules comme dans le notebook Python, via l'API des classes
`Proposition`, `Negation`, `Conjunction`, `Disjunction`, `Implication`.


In [5]:
using org.tweetyproject.logics.pl.syntax;

var a = new Proposition("a");
var b = new Proposition("b");
var c = new Proposition("c");

var f1 = a;
var f2 = new Negation(b);
var f3 = new Conjunction(a, new Negation(c));
var f4 = new Disjunction(a, b);
var f5 = new Implication(a, b);

Console.WriteLine($"a       = {a}");
Console.WriteLine($"!b      = {f2}");
Console.WriteLine($"a && !c = {f3}");
Console.WriteLine($"a || b  = {f4}");
Console.WriteLine($"a => b  = {f5}");


a       = a


!b      = !b


a && !c = a&&!c


a || b  = a||b


a => b  = (a=>b)


### 4. Parsing de formules avec `PlParser`

Le `PlParser` lit des formules depuis une chaine : `!` (negation), `&&` (conjonction),
`||` (disjonction), `=>` (implication), `<=>` (equivalence), `^^` (XOR).


In [6]:
var parser = new org.tweetyproject.logics.pl.parser.PlParser();

var fParsed = parser.parseFormula("(a || b) && !c");
Console.WriteLine($"Parse de '(a || b) && !c' = {fParsed}");

var fXor = parser.parseFormula("a ^^ b");
Console.WriteLine($"XOR 'a ^^ b' = {fXor}");


Parse de '(a || b) && !c' = (a||b)&&!c


XOR 'a ^^ b' = a^^b


### 5. Base de croyances (`PlBeliefSet`)

Une `PlBeliefSet` est un ensemble de formules (base de connaissances). On reproduit la KB
du notebook Python. **Note d'API** : IKVM expose les methodes Java en minuscules
(`kb.add(...)`, `kb.size()`) - c'est la convention Java, pas les wrappers .NET.


In [7]:
using org.tweetyproject.logics.pl.syntax;

var kb = new PlBeliefSet();
var p = new org.tweetyproject.logics.pl.parser.PlParser();
kb.add(p.parseFormula("a"));
kb.add(p.parseFormula("!b"));
kb.add(p.parseFormula("a && !c"));
kb.add(p.parseFormula("a => b"));

Console.WriteLine($"Base de croyances KB = {kb}");
Console.WriteLine($"Nombre de formules : {kb.size()}");


Base de croyances KB = { a&&!c, (a=>b), a, !b }


Nombre de formules : 4


## Conclusion

Ce notebook demontre le **port .NET fonctionnel** de Tweety pour la logique propositionnelle :

| Concept | Classe Tweety (C#) | Statut |
|---------|-------------------|--------|
| Atome propositionnel | `Proposition` | OK |
| Connecteurs | `Negation`, `Conjunction`, `Disjunction`, `Implication` | OK |
| Parsing | `PlParser.parseFormula` | OK |
| Base de croyances | `PlBeliefSet` (`.add()`, `.size()`) | OK |

**Avantage du port .NET** : aucune JVM ni JDK a installer, execution native dans le kernel
`.net-csharp`. **Limitation** : seule la logique propositionnelle (cluster `pl`) est portee
dans ce pilote ; les notebooks Python restent canoniques pour la FOL complete,
l'argumentation (notebooks 5-7), les MLN (notebook 10), etc. La semantique via
`PossibleWorld` + `SatReasoner.query` fera l'objet d'un notebook suivant.

### Exercices

Reprenez ceux du notebook Python, adaptes a l'API C# (methodes Java minuscules).


In [8]:
// Exercice 1 : Construisez la formule (a => b) && (b => a) et affichez sa forme DNF.
// Indice : PlParser + .toNnf() ou construction manuelle.
// TODO etudiant : completez ici
Console.WriteLine("Exercice 1 a completer");


Exercice 1 a completer


In [9]:
// Exercice 2 : Parsez une KB de 4 formules et affichez sa taille + contenu.
// TODO etudiant : completez ici
Console.WriteLine("Exercice 2 a completer");


Exercice 2 a completer
